<a href="https://colab.research.google.com/github/lmassaron/fine-tuning-workshop/blob/main/07_tunix_sft_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop: Tunix-Med · Part 3: Supervised Fine-Tuning with JAX & Tunix

This is the core of the workshop. We use **Tunix**, a high-performance JAX-based fine-tuning library, to train the **Gemma 3 270M** model on our cardiology dataset.

### Why JAX and Tunix?
1.  **XLA Compilation**: JAX uses the XLA compiler to fuse operations into highly efficient kernels, maximizing GPU throughput.
2.  **Streaming Data**: We implement a data pipeline that never loads the whole dataset into RAM, allowing training on huge datasets with minimal hardware.
3.  **Advanced LoRA**: We apply Low-Rank Adaptation (LoRA) not just to standard Linear layers, but also to the **Einsum** layers used in Gemma 3's attention mechanism.

### Key Concepts demonstrated:
-   **Precision**: Training in `bfloat16` for stability and speed.
-   **Memory Efficiency**: Managing JAX's aggressive memory pre-allocation.
-   **Reproducibility**: Using fixed seeds and deterministic data iterators.
-   **Surgical Checkpointing**: Saving the adapter only when the evaluation loss actually improves.

## 1 · Environment & JAX Setup

JAX by default tries to pre-allocate 75% of your GPU memory. We disable this (`XLA_PYTHON_CLIENT_PREALLOCATE=false`) to share memory with other processes (like the evaluation loops).

In [ ]:
import os
import logging

# Prevent JAX from hogging all VRAM immediately
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

# Silence verbose library logging for a cleaner workshop experience
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
for _name in ("absl", "jax", "orbax", "flax", "datasets", "transformers", "grain"):
    logging.getLogger(_name).setLevel(logging.ERROR)

import jax
import jax.numpy as jnp
import numpy as np

print(f"JAX backend : {jax.default_backend()}")
print(f"Devices     : {jax.devices()}")

## 2 · Hyperparameter Configuration

We use a conservative **Learning Rate (5e-5)** and a large **Effective Batch Size (32)**. For small models like the 270M, high learning rates can quickly lead to "Catastrophic Forgetting" or "Concept Collapse".

In [ ]:
class Config:
    MODEL_NAME = "google/gemma-3-270m-it"
    MODEL_KEY = "gemma-3-270m-it"

    DATASET_PATH = "data/medical-cardiology-qa"
    MAX_SEQ_LEN = 1024  # Fixed length allows JAX to JIT-compile the graph once
    EVAL_SPLIT = 0.1
    SEED = 42

    # LoRA Parameters
    LORA_RANK = 16  # Rank of the update matrices
    LORA_ALPHA = 32  # Scaling factor (Alpha/Rank = 2.0)
    LORA_DROPOUT = 0.05

    # Optimization
    LEARNING_RATE = 5e-5
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.1

    # Training Loop
    BATCH_SIZE = 8
    GRAD_ACCUM_STEPS = 4  # Total batch size = 8 * 4 = 32
    NUM_EPOCHS = 1  # 1 epoch is typically enough for domain specialization
    EARLY_STOP_PATIENCE = 3
    EVAL_EVERY_N_STEPS = 100
    LOG_EVERY_N_STEPS = 100
    OUTPUT_DIR = "tunix-medical-model"


params = Config()
print(f"Effective Batch Size: {params.BATCH_SIZE * params.GRAD_ACCUM_STEPS}")

## 3 · Streaming Data Pipeline

Instead of loading the whole dataset into memory (which can crash machines), we use a **lazy generator**. 
1.  Data stays on disk in **Arrow** format.
2.  We only pull the indices we need for the current batch.
3.  Tokenization happens on-the-fly.

In [ ]:
import datasets
from transformers import AutoTokenizer
from tunix.sft import peft_trainer

hf_tokenizer = AutoTokenizer.from_pretrained(params.MODEL_NAME)
full_ds = datasets.load_from_disk(params.DATASET_PATH)["train"]
n = len(full_ds)

# Create split indices
rng = np.random.default_rng(params.SEED)
all_idx = rng.permutation(n)
cut = int(n * (1.0 - params.EVAL_SPLIT))
train_idx, eval_idx = all_idx[:cut], all_idx[cut:]


def _tokenise(example):
    """Process a single row into tokens and labels (masking the user prompt)."""
    messages = example["messages"]
    full_ids = hf_tokenizer.apply_chat_template(messages, tokenize=True)
    prompt_ids = hf_tokenizer.apply_chat_template(
        messages[:-1], tokenize=True, add_generation_prompt=True
    )

    prompt_len = len(prompt_ids)
    full_ids = full_ids[: params.MAX_SEQ_LEN]

    # Label mask: 0 for prompt (ignored), 1 for answer (calculated in loss)
    mask = np.ones(len(full_ids), dtype=np.int32)
    mask[: min(prompt_len, len(full_ids))] = 0

    # Pad to fixed length for JAX JIT stability
    pad_len = params.MAX_SEQ_LEN - len(full_ids)
    tokens = np.pad(
        full_ids, (0, pad_len), constant_values=hf_tokenizer.pad_token_id
    ).astype(np.int32)
    mask = np.pad(mask, (0, pad_len), constant_values=0).astype(np.int32)
    return {"input_tokens": tokens, "input_mask": mask}


def make_data_iterator(dataset, indices, batch_size, shuffle=True, infinite=True):
    """Generator that yields JAX-compatible training inputs."""
    while True:
        idxs = indices.copy()
        if shuffle:
            np.random.shuffle(idxs)
        for start in range(0, len(idxs) - batch_size + 1, batch_size):
            batch_idx = idxs[start : start + batch_size]
            processed = [_tokenise(dataset[int(i)]) for i in batch_idx]
            tokens = np.stack([p["input_tokens"] for p in processed])
            masks = np.stack([p["input_mask"] for p in processed])
            yield peft_trainer.TrainingInput(
                input_tokens=jnp.array(tokens), input_mask=jnp.array(masks)
            )
        if not infinite:
            break


train_iter = make_data_iterator(full_ds, train_idx, params.BATCH_SIZE)
print("Streaming data pipeline configured.")

## 4 · Model & Custom LoRA Setup

Gemma 3 uses **Einsum** operations for its attention mechanism. Standard PEFT libraries often ignore these. We provide a custom `EinsumLoRALayer` to ensure we are fine-tuning the model's ability to "attend" to medical concepts, not just its MLP knowledge.

In [ ]:
from flax import nnx
from tunix.models import automodel
from huggingface_hub import snapshot_download
from tunix.sft import utils as sft_utils

# 1. Model Building
devices = np.array(jax.devices()).reshape((1, -1))
mesh = jax.sharding.Mesh(devices, axis_names=("tp", "fsdp"))
model_path = snapshot_download(params.MODEL_NAME)
model_config = automodel.call_model_config(params.MODEL_KEY)

with mesh:
    model = automodel.create_model_from_safe_tensors(
        params.MODEL_KEY, model_path, model_config, mesh, dtype=jnp.bfloat16
    )


# 2. Custom LoRA Wrapper for Einsum
class EinsumLoRALayer(nnx.Module):
    """Injects trainable low-rank matrices into Gemma's Einsum layers."""

    def __init__(
        self, einsum_module, rank, alpha, dropout, dtype=jnp.bfloat16, rngs=None
    ):
        self.einsum, self.rank, self.scale = einsum_module, rank, alpha / rank
        self.dropout = nnx.Dropout(dropout, rngs=rngs)
        w = einsum_module.w
        in_dim = np.prod(w.shape[:-1])
        out_dim = w.shape[-1]
        self.lora_a = nnx.LoRAParam(
            jax.random.normal(rngs.params(), (in_dim, rank), dtype=dtype) * 0.01
        )
        self.lora_b = nnx.LoRAParam(jnp.zeros((rank, out_dim), dtype=dtype))

    def __call__(self, x):
        base = self.einsum(x)
        delta_w = (self.lora_a[...] @ self.lora_b[...]).reshape(self.einsum.w.shape)
        return base + self.scale * jnp.einsum(
            self.einsum.einsum_str, self.dropout(x), delta_w.astype(x.dtype)
        )


def apply_lora(model, rank, alpha, dropout):
    """Walk the module graph and patch Linear and Einsum layers."""
    replaced = 0
    for path, mod in nnx.graph.iter_graph(model):
        if not path:
            continue
        parent = model
        for step in path[:-1]:
            parent = getattr(parent, str(step))
        attr = str(path[-1])
        if isinstance(mod, nnx.Linear):
            setattr(
                parent,
                attr,
                sft_utils.LinearLoRALayer(
                    mod,
                    rank,
                    alpha,
                    dropout,
                    dtype=jnp.bfloat16,
                    rngs=nnx.Rngs(params=0, dropout=0),
                ),
            )
            replaced += 1
        elif isinstance(mod, nnx.Einsum):
            setattr(
                parent,
                attr,
                EinsumLoRALayer(
                    mod,
                    rank,
                    alpha,
                    dropout,
                    dtype=jnp.bfloat16,
                    rngs=nnx.Rngs(params=0, dropout=0),
                ),
            )
            replaced += 1
    return replaced


with mesh:
    n_lora = apply_lora(model, params.LORA_RANK, params.LORA_ALPHA, params.LORA_DROPOUT)
print(f"Patched {n_lora} layers with LoRA (including Attention Einsums).")

## 5 · Training & Corrected Progress Hook

We use a custom `CleanProgressHook`. Crucially, it only saves the "Best" adapter if the **evaluation loss is lower than any previous step**. This prevents an overfitted model from being saved as the final result.

In [ ]:
import optax
import time
from tunix.sft import hooks


def export_adapter(model, params):
    """Saves LoRA weights in PEFT-compatible safetensors format."""
    # ... (full logic for mapping JAX keys to PEFT keys provided in workshop tools)
    pass


class CleanProgressHook(hooks.TrainingHooks):
    def __init__(
        self,
        log_every,
        max_steps,
        steps_per_epoch,
        model,
        params,
        export_fn,
        patience=5,
    ):
        self._log_every, self._max_steps, self._model, self._params, self._export_fn = (
            log_every,
            max_steps,
            model,
            params,
            export_fn,
        )
        self._best_loss, self._no_improve, self._t0 = float("inf"), 0, time.time()

    def on_eval_step_end(self, train_ctx, eval_loss):
        n_batches = max(1, len(eval_idx) // params.BATCH_SIZE)
        mean_loss = eval_loss / n_batches

        # The Fix: Only update 'best' if loss actually dropped
        if mean_loss < self._best_loss:
            self._best_loss = mean_loss
            self._no_improve = 0
            self._export_fn(self._model, self._params)
            print(f"  [eval] Loss improved to {mean_loss:.4f} - Adapter saved!")
        else:
            self._no_improve += 1
            print(
                f"  [eval] Loss {mean_loss:.4f} (Best: {self._best_loss:.4f}) - No improvement ({self._no_improve}/{self._patience})"
            )


# Trainer Configuration
schedule = optax.warmup_cosine_decay_schedule(
    0.0, params.LEARNING_RATE, int(params.MAX_STEPS * 0.1), params.MAX_STEPS
)
optimizer = optax.adamw(learning_rate=schedule, weight_decay=params.WEIGHT_DECAY)
trainer = peft_trainer.PeftTrainer(
    model=model,
    optimizer=optimizer,
    training_config=peft_trainer.TrainingConfig(
        max_steps=params.MAX_STEPS,
        eval_every_n_steps=params.EVAL_EVERY_N_STEPS,
        gradient_accumulation_steps=params.GRAD_ACCUM_STEPS,
    ),
)

print("Trainer ready. Starting domain specialization...")

## 6 · Execute Training

Once training starts, JAX will **JIT-compile** the entire training step. This takes about 60-90 seconds for the first batch, but every subsequent batch will run at near-hardware speed.

In [ ]:
class ReusableEvalIter:  # JAX needs fresh iterators for every eval pass
    def __iter__(self):
        return make_data_iterator(
            full_ds, eval_idx, params.BATCH_SIZE, shuffle=False, infinite=False
        )


try:
    trainer.train(train_iter, ReusableEvalIter())
except StopIteration:
    pass
print("Training complete. Best adapter is saved in ", params.OUTPUT_DIR)